# 📊 Exploratory Data Analysis — Kinh Đô FMCG Demand Forecasting

**Mục tiêu:**
1. Tổng quan dữ liệu thô và phân phối theo CATEGORY
2. Time-series plot theo CATEGORY & BRAND
3. Phân phối Total QTY (Histogram + KDE)
4. Nhận diện đỉnh nhọn Sales Spikes tại Lễ hội
5. Zero-Inflation Analysis cho TET & MOONCAKE

Lưu biểu đồ 300 dpi → `results/figures/`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Paths
BASE_DIR = os.path.abspath('..')
RAW_DIR = os.path.join(BASE_DIR, 'data', 'raw')
FIG_DIR = os.path.join(BASE_DIR, 'results', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

# Style
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'font.size': 11,
    'axes.titlesize': 14,
    'figure.dpi': 100,
})

print('Setup complete ✓')

## 1. Load & Inspect Data

In [ ]:
# Load 3 CSV files
dfs = []
for f in ['data_2023.csv', 'data_2024.csv', 'data_2025.csv']:
    path = os.path.join(RAW_DIR, f)
    dfs.append(pd.read_csv(path))
    print(f'{f}: {len(dfs[-1]):,} rows')

df = pd.concat(dfs, ignore_index=True)
df['ACTUALSHIPDATE'] = pd.to_datetime(df['ACTUALSHIPDATE'])
print(f'\nTotal: {len(df):,} rows')
print(f'Date: {df.ACTUALSHIPDATE.min().date()} → {df.ACTUALSHIPDATE.max().date()}')
print(f'\nCategories: {sorted(df.CATEGORY.unique())}')
print(f'Brands: {sorted(df.BRAND.unique())}')

df.head(10)

In [ ]:
# Basic statistics
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Descriptive Statistics ===')
df[['Total QTY', 'Total CBM']].describe()

## 2. Time-Series Plot theo CATEGORY

In [ ]:
categories = sorted(df['CATEGORY'].unique())
colors = {'DRY': '#3498DB', 'TET': '#E74C3C', 'MOONCAKE': '#F39C12', 'FRESH': '#27AE60'}

fig, axes = plt.subplots(len(categories), 1, figsize=(16, 4*len(categories)), sharex=True)
if len(categories) == 1:
    axes = [axes]

for i, cat in enumerate(categories):
    ax = axes[i]
    cat_data = df[df['CATEGORY'] == cat].groupby('ACTUALSHIPDATE')['Total QTY'].sum()
    color = colors.get(cat, '#95A5A6')
    
    ax.fill_between(cat_data.index, cat_data.values, alpha=0.3, color=color)
    ax.plot(cat_data.index, cat_data.values, color=color, linewidth=0.8)
    ax.set_title(f'Category: {cat}', fontweight='bold', fontsize=13)
    ax.set_ylabel('Total QTY')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    
    # Annotate Tet dates
    for tet in ['2023-01-22', '2024-02-10', '2025-01-29']:
        ax.axvline(pd.Timestamp(tet), color='red', linestyle=':', alpha=0.5, linewidth=1)

plt.suptitle('Time-Series Overview by CATEGORY', fontsize=16, fontweight='bold', y=1.01)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'eda_timeseries_by_category.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved → eda_timeseries_by_category.png')

## 3. Phân phối Total QTY

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram + KDE (all data)
ax = axes[0]
nonzero = df[df['Total QTY'] > 0]['Total QTY']
ax.hist(nonzero, bins=100, color='#3498DB', alpha=0.7, edgecolor='white', density=True)
nonzero.plot.kde(ax=ax, color='#E74C3C', linewidth=2)
ax.set_title('Distribution of Total QTY (nonzero)', fontweight='bold')
ax.set_xlabel('Total QTY')
ax.set_ylabel('Density')
ax.set_xlim(0, nonzero.quantile(0.99))

# Log-scale histogram
ax = axes[1]
ax.hist(np.log1p(df['Total QTY']), bins=80, color='#27AE60', alpha=0.7, edgecolor='white')
ax.set_title('Distribution of log(1 + Total QTY)', fontweight='bold')
ax.set_xlabel('log(1 + Total QTY)')
ax.set_ylabel('Count')

plt.suptitle('Total QTY Distribution Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'eda_qty_distribution.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved → eda_qty_distribution.png')

## 4. Sales Spikes — Nhận diện đỉnh Lễ hội

In [ ]:
# Daily total across all brands
daily = df.groupby('ACTUALSHIPDATE')['Total QTY'].sum().reset_index()
daily.columns = ['date', 'qty']
daily = daily.sort_values('date')

# Detect spikes: > mean + 2*std
threshold = daily['qty'].mean() + 2 * daily['qty'].std()
spikes = daily[daily['qty'] > threshold]

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(daily['date'], daily['qty'], color='#2C3E50', linewidth=0.8, alpha=0.8)
ax.scatter(spikes['date'], spikes['qty'], color='#E74C3C', s=30, zorder=5, label='Sales Spike')
ax.axhline(threshold, color='#E74C3C', linestyle='--', alpha=0.5, label=f'Threshold (μ+2σ = {threshold:,.0f})')

# Mark Tet & Mid-Autumn
tet_dates = ['2023-01-22', '2024-02-10', '2025-01-29']
ma_dates = ['2023-09-29', '2024-09-17', '2025-10-06']
for d in tet_dates:
    ax.axvline(pd.Timestamp(d), color='red', linestyle=':', alpha=0.6)
    ax.text(pd.Timestamp(d), ax.get_ylim()[1]*0.95, 'TẾT', color='red', fontsize=8, ha='center')
for d in ma_dates:
    ax.axvline(pd.Timestamp(d), color='orange', linestyle=':', alpha=0.6)
    ax.text(pd.Timestamp(d), ax.get_ylim()[1]*0.90, 'TRUNG THU', color='orange', fontsize=8, ha='center')

ax.set_title('Sales Spikes Detection — Daily Total QTY', fontweight='bold', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Total QTY')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'eda_sales_spikes.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f'Detected {len(spikes)} spike days')
print('Saved → eda_sales_spikes.png')

## 5. Zero-Inflation Analysis

In [ ]:
# Zero-inflation per category
print('=== Zero-Inflation Analysis ===')
for cat in categories:
    subset = df[df['CATEGORY'] == cat]
    zero_pct = (subset['Total QTY'] == 0).mean() * 100
    print(f'{cat:12s}: {zero_pct:5.1f}% zero rows')

# Plot seasonal brands (TET, MOONCAKE) over time
seasonal_cats = ['TET', 'MOONCAKE']
seasonal_cats = [c for c in seasonal_cats if c in df['CATEGORY'].values]

if seasonal_cats:
    fig, axes = plt.subplots(len(seasonal_cats), 1, figsize=(16, 4*len(seasonal_cats)))
    if len(seasonal_cats) == 1:
        axes = [axes]
    
    for i, cat in enumerate(seasonal_cats):
        ax = axes[i]
        cat_daily = df[df['CATEGORY'] == cat].groupby('ACTUALSHIPDATE')['Total QTY'].sum()
        
        ax.bar(cat_daily.index, cat_daily.values, width=1, color=colors.get(cat, 'gray'), alpha=0.7)
        ax.set_title(f'{cat} — Zero-Inflation Pattern', fontweight='bold')
        ax.set_ylabel('Total QTY')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        
        zero_pct = (cat_daily == 0).mean() * 100
        ax.text(0.02, 0.95, f'Zero days: {zero_pct:.1f}%', transform=ax.transAxes,
                fontsize=11, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.suptitle('Zero-Inflation in Seasonal Categories', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'eda_zero_inflation.png'), dpi=300, bbox_inches='tight')
    plt.show()
    print('Saved → eda_zero_inflation.png')

## 6. Heatmap — Doanh số theo Brand × Tháng

In [ ]:
df['month_year'] = df['ACTUALSHIPDATE'].dt.to_period('M')
pivot = df.groupby(['BRAND', 'month_year'])['Total QTY'].sum().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(20, 8))
sns.heatmap(
    np.log1p(pivot.values),
    ax=ax,
    cmap='YlOrRd',
    xticklabels=[str(c) for c in pivot.columns],
    yticklabels=pivot.index,
    cbar_kws={'label': 'log(1 + Total QTY)'},
)
ax.set_title('Brand × Month Heatmap (Log Scale)', fontweight='bold', fontsize=14)
ax.set_xlabel('Month')
ax.set_ylabel('Brand')
plt.xticks(rotation=90, fontsize=7)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'eda_brand_month_heatmap.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved → eda_brand_month_heatmap.png')

## 7. Weekly Pattern — Doanh số theo thứ trong tuần

In [ ]:
df['dayofweek'] = df['ACTUALSHIPDATE'].dt.dayofweek
dow_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

fig, ax = plt.subplots(figsize=(10, 5))
for cat in categories:
    cat_dow = df[df['CATEGORY'] == cat].groupby('dayofweek')['Total QTY'].mean()
    ax.plot(cat_dow.index, cat_dow.values, marker='o', label=cat,
            color=colors.get(cat, 'gray'), linewidth=2)

ax.set_xticks(range(7))
ax.set_xticklabels(dow_names)
ax.set_title('Average Daily QTY by Day of Week', fontweight='bold')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Avg Total QTY')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'eda_weekly_pattern.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved → eda_weekly_pattern.png')

In [ ]:
print('\n✅ EDA COMPLETE — All figures saved to results/figures/')
print(f'Directory: {FIG_DIR}')
for f in os.listdir(FIG_DIR):
    if f.startswith('eda_'):
        print(f'  • {f}')